In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import add_self_loops
from torch_geometric.nn import MessagePassing
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [3]:
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

Processing...
Done!


In [7]:
class NeutrosophicFuzzification(nn.Module):
    def __init__(self, alpha=0.5, beta=2.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta

    def forward(self, x):
        mean = x.mean(dim=0, keepdim=True)
        std = x.std(dim=0, keepdim=True) + 1e-6

        T = torch.exp(-((x - mean)**2) / (2 * std**2))
        F_val = 1 - torch.exp(-((x - mean)**2) / (2 * (self.alpha * std)**2))
        I = 1 - torch.exp(-((x - mean)**2) / (2 * (self.beta * std)**2))

        return T, I, F_val


In [ ]:
class NGNNLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='mean')

        self.W_T = nn.Linear(in_channels, out_channels)
        self.W_I = nn.Linear(in_channels, out_channels)
        self.W_F = nn.Linear(in_channels, out_channels)

    def forward(self, T, I, F_val, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=T.size(0))

        row, col = edge_index

        # 🔥 Edge feature: cosine similarity
        edge_weight = F.cosine_similarity(T[row], T[col], dim=1)

        # Transform
        T_trans = self.W_T(T)
        I_trans = self.W_I(I)
        F_trans = self.W_F(F_val)

        # Aggregate with edge weights
        T_agg = self.propagate(edge_index, x=T_trans, edge_weight=edge_weight)
        I_agg = self.propagate(edge_index, x=I_trans, edge_weight=edge_weight)
        F_agg = self.propagate(edge_index, x=F_trans, edge_weight=edge_weight)

        # Rule combination
        T_new = T_trans + T_agg
        I_new = I_trans + I_agg
        F_new = F_trans + F_agg

        return T_new, I_new, F_new

    def message(self, x_j, edge_weight):
        return edge_weight.view(-1, 1) * x_j

In [9]:
class NGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.fuzz = NeutrosophicFuzzification()

        self.layer1 = NGNNLayer(in_channels, hidden_channels)
        self.layer2 = NGNNLayer(hidden_channels, out_channels)
        # Learnable rule strengths
        self.rule_weights = nn.Parameter(torch.tensor([1.0, 1.0, 1.0]))


    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        # Step 1: Fuzzification
        T, I, F_val = self.fuzz(x)

        # Step 2: Layer 1
        T, I, F_val = self.layer1(T, I, F_val, edge_index)
        T, I, F_val = F.relu(T), F.relu(I), F.relu(F_val)

        # Step 3: Layer 2
        T, I, F_val = self.layer2(T, I, F_val, edge_index)

        # Step 4: Normalize rule strengths
        weights = torch.softmax(self.rule_weights, dim=0)

        # Step 5: Defuzzification
        H = weights[0]*T + weights[1]*I + weights[2]*F_val

        return H,F.log_softmax(H, dim=1)

In [10]:
device = torch.device('mps' if torch.cuda.is_available() else 'cpu')
model = NGNN(
    in_channels=dataset.num_node_features,
    hidden_channels=16,
    out_channels=dataset.num_classes
).to(device)

data = data.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [11]:
def train():
    model.train()
    optimizer.zero_grad()
    H,out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def test():
    model.eval()
    with torch.no_grad():
        H, out = model(data)   # 👈 unpack tuple
        
        pred = out.argmax(dim=1)

        correct = pred[data.test_mask] == data.y[data.test_mask]
        acc = int(correct.sum()) / int(data.test_mask.sum())

    return acc

In [12]:
for epoch in range(1, 201):
    loss = train()
    acc = test()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}, Test Accuracy: {acc:.4f}")

Epoch 10, Loss: 1.1251, Test Accuracy: 0.4620
Epoch 20, Loss: 0.2178, Test Accuracy: 0.7720
Epoch 30, Loss: 0.0352, Test Accuracy: 0.7510
Epoch 40, Loss: 0.0095, Test Accuracy: 0.7810
Epoch 50, Loss: 0.0059, Test Accuracy: 0.7850
Epoch 60, Loss: 0.0052, Test Accuracy: 0.7760
Epoch 70, Loss: 0.0060, Test Accuracy: 0.7770
Epoch 80, Loss: 0.0072, Test Accuracy: 0.7810
Epoch 90, Loss: 0.0081, Test Accuracy: 0.7830
Epoch 100, Loss: 0.0085, Test Accuracy: 0.7870
Epoch 110, Loss: 0.0085, Test Accuracy: 0.7910
Epoch 120, Loss: 0.0083, Test Accuracy: 0.7920
Epoch 130, Loss: 0.0079, Test Accuracy: 0.7930
Epoch 140, Loss: 0.0076, Test Accuracy: 0.7930
Epoch 150, Loss: 0.0072, Test Accuracy: 0.7930
Epoch 160, Loss: 0.0069, Test Accuracy: 0.7930
Epoch 170, Loss: 0.0067, Test Accuracy: 0.7920
Epoch 180, Loss: 0.0064, Test Accuracy: 0.7910
Epoch 190, Loss: 0.0061, Test Accuracy: 0.7950
Epoch 200, Loss: 0.0059, Test Accuracy: 0.7950


In [13]:
def get_embeddings():
    model.eval()
    with torch.no_grad():
        H, out = model(data)
    return H

In [14]:
H = get_embeddings()

print("Shape of X_new:", H.shape)
print("\nFirst 5 nodes (X_new):\n", H[:5])

Shape of X_new: torch.Size([2708, 7])

First 5 nodes (X_new):
 tensor([[-2.1644, -2.1007, -1.9549,  4.2368, -1.9295, -2.2391, -2.6052],
        [-0.9700, -0.9845, -3.5889, -2.7342,  7.6150, -4.4864, -1.7983],
        [-0.8508, -2.2592, -3.0929, -0.3577,  5.2348, -3.6287, -2.0593],
        [ 6.3399, -0.5722, -4.2826, -2.2385, -0.8954, -1.9163,  0.0981],
        [-2.8575, -2.2388, -1.6621,  4.2391, -1.8760, -1.8525, -2.7524]])


In [15]:
def get_predictions():
    model.eval()
    with torch.no_grad():
        H, out = model(data)
        pred = out.argmax(dim=1)
    return pred

In [16]:
pred = get_predictions()

print("First 20 nodes classification:\n")
for i in range(50):
    print(f"Node {i}: Predicted = {pred[i].item()}, Actual = {data.y[i].item()}")

First 20 nodes classification:

Node 0: Predicted = 3, Actual = 3
Node 1: Predicted = 4, Actual = 4
Node 2: Predicted = 4, Actual = 4
Node 3: Predicted = 0, Actual = 0
Node 4: Predicted = 3, Actual = 3
Node 5: Predicted = 2, Actual = 2
Node 6: Predicted = 0, Actual = 0
Node 7: Predicted = 3, Actual = 3
Node 8: Predicted = 3, Actual = 3
Node 9: Predicted = 2, Actual = 2
Node 10: Predicted = 0, Actual = 0
Node 11: Predicted = 0, Actual = 0
Node 12: Predicted = 4, Actual = 4
Node 13: Predicted = 3, Actual = 3
Node 14: Predicted = 3, Actual = 3
Node 15: Predicted = 3, Actual = 3
Node 16: Predicted = 2, Actual = 2
Node 17: Predicted = 3, Actual = 3
Node 18: Predicted = 1, Actual = 1
Node 19: Predicted = 3, Actual = 3
Node 20: Predicted = 5, Actual = 5
Node 21: Predicted = 3, Actual = 3
Node 22: Predicted = 4, Actual = 4
Node 23: Predicted = 6, Actual = 6
Node 24: Predicted = 3, Actual = 3
Node 25: Predicted = 3, Actual = 3
Node 26: Predicted = 6, Actual = 6
Node 27: Predicted = 3, Actual = 

In [17]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

def evaluate_metrics():
    model.eval()
    with torch.no_grad():
        H, out = model(data)
        pred = out.argmax(dim=1).cpu()
        true = data.y.cpu()

        # Only consider test nodes
        test_idx = data.test_mask.cpu()
        y_true = true[test_idx]
        y_pred = pred[test_idx]

        # Compute metrics
        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, average='macro')  # macro = average over classes
        precision = precision_score(y_true, y_pred, average='macro')
        recall = recall_score(y_true, y_pred, average='macro')
        cm = confusion_matrix(y_true, y_pred)

    return acc, precision, recall, f1, cm

In [19]:
data.y[20].item()

5

In [20]:
# Make predictions
model.eval()
with torch.no_grad():
    H, out = model(data)
    pred = out.argmax(dim=1)

# Only consider test nodes
test_idx = data.test_mask.nonzero(as_tuple=True)[0]
y_true = data.y[test_idx]
y_pred = pred[test_idx]

# Find misclassified nodes
wrong_idx = test_idx[y_true != y_pred]

print(f"Total wrong predictions in test set: {len(wrong_idx)}\n")
print("List of misclassified nodes (Node ID : Predicted -> Actual):")
for node in wrong_idx:
    print(f"Node {node.item()}: Pred = {y_pred[node == test_idx].item()}, Actual = {y_true[node == test_idx].item()}")

Total wrong predictions in test set: 205

List of misclassified nodes (Node ID : Predicted -> Actual):
Node 1708: Pred = 2, Actual = 3
Node 1728: Pred = 2, Actual = 3
Node 1741: Pred = 0, Actual = 3
Node 1743: Pred = 1, Actual = 2
Node 1764: Pred = 2, Actual = 5
Node 1774: Pred = 1, Actual = 4
Node 1792: Pred = 5, Actual = 3
Node 1793: Pred = 5, Actual = 4
Node 1796: Pred = 0, Actual = 4
Node 1799: Pred = 1, Actual = 3
Node 1801: Pred = 1, Actual = 0
Node 1802: Pred = 1, Actual = 3
Node 1803: Pred = 6, Actual = 0
Node 1807: Pred = 4, Actual = 3
Node 1834: Pred = 2, Actual = 1
Node 1840: Pred = 0, Actual = 5
Node 1844: Pred = 6, Actual = 3
Node 1845: Pred = 0, Actual = 5
Node 1854: Pred = 3, Actual = 1
Node 1869: Pred = 4, Actual = 3
Node 1871: Pred = 1, Actual = 3
Node 1878: Pred = 1, Actual = 5
Node 1892: Pred = 4, Actual = 0
Node 1893: Pred = 4, Actual = 3
Node 1905: Pred = 4, Actual = 0
Node 1907: Pred = 3, Actual = 4
Node 1908: Pred = 0, Actual = 4
Node 1909: Pred = 3, Actual = 0
N